# AI Debugger Pro — Vulnerability Classifier Training

Reproduces the actual pipeline behind the deployed `model.pkl`:
TF-IDF (character n-grams) + Logistic Regression, **binary** classification (`clean` vs `vulnerable`), trained on `code_vul_dataset.csv` + `secure_programming_dpo.json`.

Runs top to bottom in Google Colab with no manual edits required beyond Section 3.

## 1. Install Dependencies

In [ ]:
!pip install -q scikit-learn==1.6.1 pandas joblib


## 2. Import Libraries

In [ ]:
import os
import re
import sys
import json
import platform
import warnings

import numpy as np
import pandas as pd
import sklearn
import joblib
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    train_test_split, cross_val_score, learning_curve, validation_curve
)
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve
)

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


## 3. Dataset Setup

Two options: mount Google Drive, or upload the two files directly. **Edit `DATASET_DIR` if using Drive.**

In [ ]:
USE_DRIVE = False  # <-- set True if your files are in Google Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATASET_DIR = "/content/drive/MyDrive/ai_debugger_pro"  # <-- ADJUST THIS PATH
else:
    from google.colab import files
    print("Upload code_vul_dataset.csv and secure_programming_dpo.json")
    uploaded = files.upload()
    DATASET_DIR = "."

CSV_PATH = os.path.join(DATASET_DIR, "code_vul_dataset.csv")
JSON_PATH = os.path.join(DATASET_DIR, "secure_programming_dpo.json")

assert os.path.exists(CSV_PATH), f"Missing file: {CSV_PATH}"
assert os.path.exists(JSON_PATH), f"Missing file: {JSON_PATH}"
print("Found both dataset files.")


## 4. Explore Dataset

In [ ]:
df_csv_raw = pd.read_csv(CSV_PATH)
print("CSV rows:", len(df_csv_raw))
print("CSV columns:", list(df_csv_raw.columns))
print("\nCSV label distribution:")
print(df_csv_raw['label'].value_counts())
print("\nMissing values per column:")
print(df_csv_raw.isna().sum())

json_rows_raw = [json.loads(l) for l in open(JSON_PATH, encoding='utf-8') if l.strip()]
print(f"\nJSON rows: {len(json_rows_raw)}")
print("JSON keys:", list(json_rows_raw[0].keys()))
from collections import Counter
print("\nJSON language distribution:")
print(Counter(r['lang'] for r in json_rows_raw))

print("\nExample CSV row:")
print(df_csv_raw.iloc[0][['label', 'code']])
print("\nExample JSON row (truncated):")
ex = json_rows_raw[0]
print({k: (v[:120] + '...' if isinstance(v, str) and len(v) > 120 else v) for k, v in ex.items()})


## 5. Data Preprocessing

Same cleaning used to produce the deployed model: strip markdown code fences, **strip comments** (critical — without this the model reads English words like 'vulnerable' in comments instead of learning code patterns), filter to the languages the debugger supports, collapse CWE labels into `clean`/`vulnerable`, then deduplicate across both sources.

In [ ]:
KEEP_LANGS = {"javascript", "php", "python", "sql", "html"}

# Collapse the 14 CWE labels in the CSV into a binary scheme.
# Memory/pointer bugs (C-style) are dropped: irrelevant to a JS/PHP/Python web debugger.
VULN_CWE_LABELS = {
    'Improper Neutralization of Special Elements used in an SQL Command (\u201cSQL Injection\u201d)',
    'Improper Neutralization of Input During Web Page Generation (\u201cCross-site Scripting\u201d)',
    'Improper Control of Generation of Code (\u201cCode Injection\u201d)',
    'Improper Neutralization of Special Elements used in an OS Command (\u201cOS Command Injection\u201d)',
    'Deserialization of Untrusted Data',
    'Improper Input Validation',
    'Improper Limitation of a Pathname to a Restricted Directory (\u201cPath Traversal\u201d)',
    'URL Redirection to Untrusted Site (\u201cOpen Redirect\u201d)',
    'Improper Restriction of XML External Entity Reference',
    'Improper Output Neutralization for Logs',
}
DROPPED_CWE_LABELS = {
    'Out-of-bounds Write', 'NULL Pointer Dereference', 'Integer Overflow or Wraparound'
}

def strip_fences(code):
    """Remove ```lang ... ``` markdown wrappers; return (code, lang)."""
    if not isinstance(code, str):
        return "", None
    m = re.match(r"^\s*```(\w+)?\s*\n(.*?)\n?\s*```\s*$", code, re.DOTALL)
    if m:
        return m.group(2), (m.group(1) or "").lower()
    return code, None

def strip_comments(code):
    """Remove comments so the model can't shortcut on label-leaking text."""
    code = re.sub(r"/\*.*?\*/", "", code, flags=re.DOTALL)
    code = re.sub(r"//[^\n]*", "", code)
    code = re.sub(r"^\s*#[^\n]*$", "", code, flags=re.M)
    code = re.sub(r"<!--.*?-->", "", code, flags=re.DOTALL)
    code = re.sub(r"\n\s*\n+", "\n", code)
    return code.strip()

def clean(code):
    body, lang = strip_fences(code)
    return strip_comments(body), lang

# --- CSV: apply binary label mapping, language filter, cleaning ---
csv_rows = []
for _, r in df_csv_raw.iterrows():
    if r['label'] == 'safe':
        label = 'clean'
    elif r['label'] in VULN_CWE_LABELS:
        label = 'vulnerable'
    else:
        continue  # dropped CWE (irrelevant language/domain)
    body, lang = clean(r['code'])
    if lang not in KEEP_LANGS or len(body) < 20:
        continue
    csv_rows.append({'code': body, 'label': label, 'lang': lang, 'source': 'csv'})
part_a = pd.DataFrame(csv_rows)

# --- JSON: chosen=safe, rejected=vulnerable (DPO pairs) ---
json_rows = []
for r in json_rows_raw:
    lang = r.get('lang', '').lower()
    if lang not in KEEP_LANGS:
        continue
    safe_code, _ = clean(r.get('chosen', ''))
    vuln_code, _ = clean(r.get('rejected', ''))
    if len(safe_code) >= 20:
        json_rows.append({'code': safe_code, 'label': 'clean', 'lang': lang, 'source': 'json'})
    if len(vuln_code) >= 20:
        json_rows.append({'code': vuln_code, 'label': 'vulnerable', 'lang': lang, 'source': 'json'})
part_b = pd.DataFrame(json_rows)

# --- merge + dedupe (prevents train/test leakage from near-duplicate snippets) ---
data = pd.concat([part_a, part_b], ignore_index=True)
before = len(data)
data = data.drop_duplicates(subset=['code']).reset_index(drop=True)
print(f"CSV usable: {len(part_a)} | JSON usable: {len(part_b)}")
print(f"Merged: {before} -> {len(data)} after removing duplicates")
print("\nFinal label distribution:")
print(data['label'].value_counts())
print("\nFinal language distribution:")
print(data['lang'].value_counts())


## 6. Feature Extraction

Character n-grams (2–5 chars), not word tokens — code is full of symbols (`$_GET[`, `" . $`, `innerHTML=`) that word tokenizers destroy but char n-grams capture.

In [ ]:
X_all = data['code']
y_all = data['label']

demo_vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), max_features=50000)
demo_matrix = demo_vectorizer.fit_transform(X_all)
print("Input: raw code string, e.g.:")
print(repr(X_all.iloc[0][:80]), "...")
print("\nFeature matrix shape (n_samples, n_features):", demo_matrix.shape)
print("Example feature vector shape for one sample:", demo_matrix[0].shape)
print("Non-zero features in that sample:", demo_matrix[0].nnz)


## 7. Dataset Splitting

Stratified 80/10/10 train/validation/test split. No subject/session grouping applies here (each row is an independent code snippet, already deduplicated above), so a standard stratified split is appropriate — group-aware splitting is not needed.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X_all, y_all, test_size=0.2, random_state=RANDOM_SEED, stratify=y_all
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=RANDOM_SEED, stratify=y_temp
)

print(f"Training samples:   {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Testing samples:    {len(X_test)}")

# Leakage check: confirm no identical code string appears in more than one split
train_set, val_set, test_set = set(X_train), set(X_val), set(X_test)
overlap = (train_set & val_set) | (train_set & test_set) | (val_set & test_set)
print(f"\nOverlapping samples across splits: {len(overlap)} (should be 0)")


## 8. Model Creation

TF-IDF + Logistic Regression pipeline, matching the deployed model's structure (`Pipeline` with steps `tfidf` and `classifier`).

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=(2, 5),
        min_df=2,
        sublinear_tf=True,
        max_features=50000,
        lowercase=True,
    )),
    ('classifier', LogisticRegression(
        max_iter=3000,
        C=5.0,
        class_weight='balanced',   # dataset is imbalanced (more 'clean' than 'vulnerable')
        random_state=RANDOM_SEED,
    )),
])

print(pipeline)
print("\nClassifier hyperparameters:")
for k, v in pipeline.named_steps['classifier'].get_params().items():
    print(f"  {k}: {v}")


## 9. Model Training

Logistic Regression is a convex optimization, not an epoch-based training loop — `max_iter` is the equivalent stopping control, and scikit-learn already applies its own internal early stopping once the solver converges. Fit on the training set only.

In [ ]:
import time
t0 = time.time()
pipeline.fit(X_train, y_train)
print(f"Trained in {time.time() - t0:.1f}s")
print(f"Solver converged in {pipeline.named_steps['classifier'].n_iter_[0]} iterations "
      f"(max_iter={pipeline.named_steps['classifier'].max_iter})")

val_acc = pipeline.score(X_val, y_val)
train_acc = pipeline.score(X_train, y_train)
print(f"\nTrain accuracy:      {train_acc:.1%}")
print(f"Validation accuracy: {val_acc:.1%}")


## 10. Training Visualization

No per-epoch loss curve exists for this model type. The equivalents used here: a **learning curve** (does more data help?) and a **validation curve** (is the regularization strength `C` well chosen?).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Learning curve: accuracy vs. training set size ---
train_sizes, train_scores, val_scores = learning_curve(
    pipeline, X_train, y_train, cv=5, random_state=RANDOM_SEED,
    train_sizes=np.linspace(0.2, 1.0, 5), n_jobs=-1,
)
axes[0].plot(train_sizes, train_scores.mean(axis=1), 'o-', label='Training accuracy')
axes[0].plot(train_sizes, val_scores.mean(axis=1), 'o-', label='CV validation accuracy')
axes[0].set_xlabel('Training set size')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Learning Curve')
axes[0].legend()
axes[0].grid(alpha=0.3)

# --- Validation curve: accuracy vs. regularization strength C ---
C_range = [0.1, 0.5, 1, 2, 5, 10, 20]
train_scores_c, val_scores_c = validation_curve(
    Pipeline([('tfidf', pipeline.named_steps['tfidf']),
              ('classifier', LogisticRegression(max_iter=3000, class_weight='balanced',
                                                 random_state=RANDOM_SEED))]),
    X_train, y_train, param_name='classifier__C', param_range=C_range, cv=5, n_jobs=-1,
)
axes[1].plot(C_range, train_scores_c.mean(axis=1), 'o-', label='Training accuracy')
axes[1].plot(C_range, val_scores_c.mean(axis=1), 'o-', label='CV validation accuracy')
axes[1].set_xscale('log')
axes[1].set_xlabel('C (inverse regularization strength)')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Validation Curve')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()


## 11. Evaluation (Test Set Only)

In [ ]:
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, list(pipeline.classes_).index('vulnerable')]

print("Classification Report (test set):\n")
print(classification_report(y_test, y_pred, digits=3))

auc = roc_auc_score((y_test == 'vulnerable').astype(int), y_proba)
print(f"ROC-AUC: {auc:.3f}")

# Leakage sanity check: top features should be CODE patterns, not English words
# like 'vulnerable' or 'unsafe' -- if you see those, something in preprocessing failed.
names = pipeline.named_steps['tfidf'].get_feature_names_out()
coefs = pipeline.named_steps['classifier'].coef_[0]
top_vuln = sorted(zip(coefs, names), reverse=True)[:10]
print("\nTop features pushing toward 'vulnerable' (should look like code, not English):")
for score, name in top_vuln:
    print(f"  {score:6.2f}  {name!r}")


## 12. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=pipeline.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=pipeline.classes_)
fig, ax = plt.subplots(figsize=(5, 5))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
plt.title('Confusion Matrix — Test Set')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()


## 13. Model Performance Summary

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

pos_label = 'vulnerable'
print("=== TEST SET PERFORMANCE ===")
print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.1%}")
print(f"Precision:     {precision_score(y_test, y_pred, pos_label=pos_label):.1%}")
print(f"Recall:        {recall_score(y_test, y_pred, pos_label=pos_label):.1%}")
print(f"F1-Score:      {f1_score(y_test, y_pred, pos_label=pos_label):.1%}")
print(f"ROC-AUC:       {auc:.3f}")


## 14. Save Model

Saved with `joblib`, matching how the existing `app.py` loads it (`joblib.load('model.pkl')`) — no backend code changes needed.

In [ ]:
MODEL_FILENAME = "vulnerability_pipeline.pkl"
joblib.dump(pipeline, MODEL_FILENAME)
print(f"Saved: {MODEL_FILENAME}")


## 15. Download Model

In [ ]:
from google.colab import files
files.download(MODEL_FILENAME)


## 16. Reproducibility

In [ ]:
print("=== REPRODUCIBILITY INFO ===")
print(f"Python version:      {platform.python_version()}")
print(f"scikit-learn version:{sklearn.__version__}")
print(f"pandas version:      {pd.__version__}")
print(f"numpy version:       {np.__version__}")
print(f"Random seed:         {RANDOM_SEED}")
print(f"Total samples:       {len(data)}")
print(f"Classes:             {list(pipeline.classes_)}")
print(f"Class distribution:  {dict(data['label'].value_counts())}")
print(f"Final model config:  {pipeline.named_steps['classifier'].get_params()}")


## 17. Final Verification

Load the saved model back from disk (as `app.py` would) and predict on one real test-set sample, confirming the round trip works end to end.

In [ ]:
loaded_model = joblib.load(MODEL_FILENAME)

sample_idx = 0
sample_code = X_test.iloc[sample_idx]
actual_label = y_test.iloc[sample_idx]

pred_label = loaded_model.predict([sample_code])[0]
pred_conf = max(loaded_model.predict_proba([sample_code])[0])

print("Input (first 200 chars):")
print(sample_code[:200])
print(f"\nPredicted Class: {pred_label}")
print(f"Confidence:      {pred_conf:.1%}")
print(f"Actual Class:    {actual_label}")
print(f"Match:           {pred_label == actual_label}")
